<a href="https://colab.research.google.com/github/noorfatima524-ai/Agentic-AI-Assistant/blob/main/Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Upload and Extract Project
import os, zipfile, shutil, glob
from google.colab import files

uploaded = files.upload()
print('Uploaded files:', list(uploaded.keys()))

zip_files = glob.glob('/content/*.zip')
if not zip_files:
    raise FileNotFoundError('Please upload the TrendPilot ZIP file first.')

zip_path = zip_files[0]
project_dir = '/content/TrendPilot_Task2_Complete'

if os.path.exists(project_dir):
    shutil.rmtree(project_dir)
os.makedirs(project_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(project_dir)

print('Project extracted to:', project_dir)
for root, dirs, files_ in os.walk(project_dir):
    level = root.replace(project_dir, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    for f in files_[:10]:
        print(f'{indent}  {f}')

Saving TrendPilot_Task2_Complete.zip to TrendPilot_Task2_Complete.zip
Uploaded files: ['TrendPilot_Task2_Complete.zip']
Project extracted to: /content/TrendPilot_Task2_Complete
TrendPilot_Task2_Complete/
  TrendPilot_Task2_Complete/
    README.md
    demo.py
    requirements.txt
    tests/
      test_cases.md
    report/
    screenshots/
    app/
      memory.py
      __init__.py
      main.py
      prompts.py
      agent.py
      tools.py
    outputs/
      saved_results/
      generated_scripts/
      generated_posts/
        sample_yolov8_linkedin.md


In [8]:
# Cell 2: Dynamically locate requirements.txt and install dependencies
import os, glob, subprocess

# Search for requirements.txt inside the project directory
req_matches = glob.glob('/content/TrendPilot_Task2_Complete/**/requirements.txt', recursive=True)

if req_matches:
    req_path = req_matches[0]
    print(f"✅ Found requirements file at: {req_path}")
    !%pip install -q -r {req_path}
else:
    print("⚠️ Warning: requirements.txt not found in extracted directory. Installing core defaults...")

# Install guaranteed required packages
!%pip install -q streamlit pyngrok requests

✅ Found requirements file at: /content/TrendPilot_Task2_Complete/TrendPilot_Task2_Complete/requirements.txt
/bin/bash: line 1: fg: no job control
/bin/bash: line 1: fg: no job control


In [28]:
import os, time, subprocess

# 1. Kill stale Ollama instances
!pkill -f ollama || true
time.sleep(2)

# 2. Set environment variables for longer timeouts
os.environ['OLLAMA_KEEP_ALIVE'] = '30m'

# 3. Start Ollama server in background
!nohup ollama serve > /dev/null 2>&1 &
time.sleep(5)

# 4. Pull ultra-fast 2B model optimized for fast responses
!ollama pull gemma:2b

print("✅ Ollama restarted with GPU & extended keep-alive settings.")

^C

✅ Ollama restarted with GPU & extended keep-alive settings.


In [11]:
# Cell 4: Locate Application Folder
import os

candidate_dirs = [
    '/content/TrendPilot_Task2_Complete',
    '/content/TrendPilot_Task2_Complete/TrendPilot_Task2_Complete'
]

APP_DIR = None
for d in candidate_dirs:
    if os.path.exists(os.path.join(d, 'app', 'main.py')):
        APP_DIR = d
        break
    if os.path.exists(os.path.join(d, 'main.py')):
        APP_DIR = d
        break

if APP_DIR is None:
    raise FileNotFoundError('Could not locate app/main.py or main.py in the extracted project.')

os.chdir(APP_DIR)
print('Application directory:', APP_DIR)
print('Files:', os.listdir(APP_DIR))

Application directory: /content/TrendPilot_Task2_Complete/TrendPilot_Task2_Complete
Files: ['tests', 'README.md', 'report', 'screenshots', 'app', 'outputs', 'demo.py', 'requirements.txt']


In [12]:
# Cell 5: Import Check
import os, sys
sys.path.insert(0, APP_DIR)

if os.path.exists(os.path.join(APP_DIR, 'app', 'main.py')):
    print('Detected package-based project structure.')
elif os.path.exists(os.path.join(APP_DIR, 'main.py')):
    print('Detected root-level main.py structure.')
else:
    print('Please inspect the extracted files before continuing.')

Detected package-based project structure.


In [17]:
# Cell 6: Terminate existing Streamlit processes and launch fresh instance
import os, subprocess, time

# 1. Kill any existing Streamlit processes running on port 8501
!fuser -k 8501/tcp > /dev/null 2>&1 || pkill -f streamlit || true
time.sleep(2)

# 2. Set up logging and launch Streamlit
streamlit_file = 'app/main.py' if os.path.exists('app/main.py') else 'main.py'
log_path = '/content/trendpilot_streamlit.log'

log_file = open(log_path, 'w')

process = subprocess.Popen(
    [
        'streamlit', 'run', streamlit_file,
        '--server.port', '8501',
        '--server.address', '0.0.0.0',
        '--server.headless', 'true'
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(7)
print('Streamlit process started with PID:', process.pid)

# 3. Output logs to verify clean startup
with open(log_path, 'r') as f:
    print('\n--- Streamlit Log Output ---')
    print(f.read()[-2000:])

Streamlit process started with PID: 5978

--- Streamlit Log Output ---


2026-09-18 06:28:13.567 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.247.160.47:8501

2026-09-18 06:28:17.601 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-kkb-ass1c2-1pr57r4jet2em-c.asia-southeast1-2.prod.colab.dev, host=m-s-kkb-ass1c2-1pr57r4jet2em.asia-southeast1-c.c.codatalab-user-runtimes.internal:8007



In [18]:
# Cell 7: Expose Streamlit Port via Colab Proxy or localtunnel

# Option 1: Native Google Colab Port Proxy (Recommended)
from google.colab import output
from google.colab.output import eval_js

# Generate official Colab proxy URL for port 8501
try:
    proxy_url = eval_js("google.colab.kernel.proxyPort(8501)")
    print("--------------------------------------------------")
    print("🚀 Access TrendPilot App Here:")
    print(proxy_url)
    print("--------------------------------------------------")
except Exception as e:
    print("Could not generate Colab proxy URL:", e)

# Option 2: Localtunnel fallback (If proxy link demands auth)
# Run in shell if needed: !npx localtunnel --port 8501

--------------------------------------------------
🚀 Access TrendPilot App Here:
https://8501-m-s-kkb-ass1c2-1pr57r4jet2em-c.asia-southeast1-2.prod.colab.dev
--------------------------------------------------


In [16]:
# Cell 8: Verify Output Files
import os

print('Project directory:', APP_DIR)
print('Output-related folders/files:')
for root, dirs, files_ in os.walk(APP_DIR):
    for f in files_:
        if 'output' in root.lower() or 'memory' in f.lower() or f.endswith('.md'):
            print(os.path.join(root, f))

Project directory: /content/TrendPilot_Task2_Complete/TrendPilot_Task2_Complete
Output-related folders/files:
/content/TrendPilot_Task2_Complete/TrendPilot_Task2_Complete/README.md
/content/TrendPilot_Task2_Complete/TrendPilot_Task2_Complete/tests/test_cases.md
/content/TrendPilot_Task2_Complete/TrendPilot_Task2_Complete/app/memory.py
/content/TrendPilot_Task2_Complete/TrendPilot_Task2_Complete/outputs/generated_posts/sample_yolov8_linkedin.md


In [19]:
# Install localtunnel
!npm install -g localtunnel

# Get your Colab IP address (you will need this password to open localtunnel)
import urllib.request
ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("="*50)
print("🔑 YOUR TUNNEL PASSWORD (IP):", ip)
print("="*50)

# Start tunnel on port 8501
import subprocess
tunnel_process = subprocess.Popen(["npx", "localtunnel", "--port", "8501"])

# Wait 3 seconds and fetch tunnel URL
import time; time.sleep(3)

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 4s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴==================================================
🔑 YOUR TUNNEL PASSWORD (IP): 35.247.160.47


In [20]:
!npx localtunnel --port 8501

⠙⠹⠸⠼your url is: https://sweet-carrots-clap.loca.lt
^C


In [30]:
import os, subprocess, time, re

# 1. Download and set permissions for cloudflared
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

# 2. Kill existing Streamlit and Cloudflare processes
!fuser -k 8501/tcp > /dev/null 2>&1 || pkill -f streamlit || pkill -f cloudflared || true
time.sleep(2)

# 3. Configure working directory and paths
if 'APP_DIR' not in globals() or not APP_DIR or not os.path.exists(APP_DIR):
    candidate_dirs = [
        '/content/TrendPilot_Task2_Complete',
        '/content/TrendPilot_Task2_Complete/TrendPilot_Task2_Complete'
    ]
    for d in candidate_dirs:
        if os.path.exists(os.path.join(d, 'app', 'main.py')) or os.path.exists(os.path.join(d, 'main.py')):
            APP_DIR = d
            break

os.environ['PYTHONPATH'] = APP_DIR
os.chdir(APP_DIR)

streamlit_file = 'app/main.py' if os.path.exists('app/main.py') else 'main.py'
log_path = '/content/trendpilot_streamlit.log'

# 4. Launch Streamlit
log_file = open(log_path, 'w')
st_process = subprocess.Popen(
    [
        'streamlit', 'run', streamlit_file,
        '--server.port', '8501',
        '--server.address', '0.0.0.0',
        '--server.headless', 'true',
        '--server.enableCORS', 'false',
        '--server.enableXsrfProtection', 'false'
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

# 5. Launch Cloudflare Tunnel
print("Starting Cloudflare Tunnel...")
cf_process = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# 6. Retrieve public URL
time.sleep(4)
url_found = False
for _ in range(12):
    line = cf_process.stderr.readline()
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            print("=" * 60)
            print("🚀 OPEN TRENDPILOT HERE:")
            print(match.group(0))
            print("=" * 60)
            url_found = True
            break
    time.sleep(1)

if not url_found:
    print("❌ Link creation delayed. Run: !cat /content/trendpilot_streamlit.log")

/content/cloudflared: Text file busy
Starting Cloudflare Tunnel...
🚀 OPEN TRENDPILOT HERE:
https://normally-visitors-yea-pizza.trycloudflare.com


In [31]:
import time, subprocess

# 1. Terminate any existing stalled Ollama processes
!pkill -f ollama || true
time.sleep(2)

# 2. Start the Ollama server silently in the background
!nohup ollama serve > /dev/null 2>&1 &
time.sleep(5)

# 3. Pull/Verify model availability (gemma:2b)
!ollama pull gemma:2b

# 4. Verify local API response
import requests
try:
    res = requests.get('http://localhost:11434/api/tags')
    if res.status_code == 200:
        print("\n✅ Ollama is up and running! Installed models:")
        print([m['name'] for m in res.json().get('models', [])])
    else:
        print("\n⚠️ Ollama returned status code:", res.status_code)
except Exception as e:
    print("\n❌ Ollama connection test failed:", e)

^C


✅ Ollama is up and running! Installed models:
['gemma:2b', 'gemma3:4b']
